In [78]:
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import time
import pandas as pd
import keyboard

# Pengaturan webdriver ===================================================================================

# ====== INISIALISASI OPTIONS ======
edge_options = Options()
edge_options.add_argument(r"user-data-dir=D:\Tools\selenium_edge_profile")
edge_options.add_argument("profile-directory=Stockbit")
edge_options.add_argument('--disable-blink-features=AutomationControlled')
edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])
edge_options.add_experimental_option('useAutomationExtension', False)

Path_Webdriver = "./Driver/msedgedriver.exe"

# ====== INISIALISASI WEBDRIVER ======
service = Service(executable_path=Path_Webdriver)
driver = webdriver.Edge(options=edge_options)

SAHAM = "BBCA.JK"

driver.get("https://stockbit.com/login")
time.sleep(3)
try:
    driver.get(link)
    print("Berhasil membuka Stockbit.")
    
    # Mencari input pencarian
    search = WebDriverWait(driver, 30).until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, '[data-cy="top-navbar-search-input-desktop"]'))
    )
    search.click()
    search.send_keys(SAHAM.split(".")[0] + Keys.ENTER)
    print(f"Berhasil mencari saham {SAHAM}.")
except Exception as e:
    print(f"Gagal mencari saham: {e}")


Berhasil membuka Stockbit.
Berhasil mencari saham BBCA.JK.


In [79]:
# Menemukan dan membuka tab Broker Summary
try:
    element = WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.XPATH, "//p[contains(text(), 'Broker Summary')]"))
    )
    print("Tab Broker Summary ditemukan.")
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", element)
    time.sleep(1.5)

    # Cari container Broker Summary dulu (elemen section-nya, bukan cuma teks judulnya)
    broker_section = driver.find_element(
        By.XPATH, "//p[contains(text(), 'Broker Summary')]/ancestor::div[2]"
    )

    # Baru cari date_button DI DALAM section itu saja
    date_button = broker_section.find_element(
        By.XPATH,
        ".//button[@aria-label='Previous day']/following-sibling::button[1]"
    )

    # Scroll dengan offset biar gak ketutup sticky navbar
    driver.execute_script("""
        const el = arguments[0];
        const rect = el.getBoundingClientRect();
        window.scrollBy(0, rect.top - 150);
    """, date_button)
    time.sleep(0.5)

    # Langsung JS click, skip native click
    driver.execute_script("arguments[0].click();", date_button)
    time.sleep(2)
except Exception as e:
    print(f"Gagal menemukan/membuka Broker Summary: {e}")

Tab Broker Summary ditemukan.


In [80]:
import pandas as pd

# Tambahkan parse_dates=['NamaKolom']
df = pd.read_csv("./Dataset/BBCA.JK.csv", parse_dates=['Date'])

year = df['Date'].iloc[0].year
month = df['Date'].iloc[0].month
day = df['Date'].iloc[0].day

print(year)
print(month)
print(day)


2023
7
3


In [81]:
# Khusus cari tanggal

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# Pastikan popup date picker udah muncul
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "react-datepicker"))
)
time.sleep(0.5)

# Tes set ke tanggal tertentu, misal tanggal 2 (yang masih di bulan yang sama/aktif)
target_day = 3
day_str = f"{target_day:03d}"  # jadi "002"

day_cell = driver.find_element(
    By.XPATH,
    f"//div[contains(@class,'react-datepicker__day--{day_str}') "
    f"and not(contains(@class,'outside-month')) "
    f"and not(contains(@class,'disabled'))]"
)

print(f"Ketemu cell tanggal {target_day}, teks: {day_cell.text}")
driver.execute_script("arguments[0].click();", day_cell)
time.sleep(1)

# Re-find broker_section dulu (yang lama udah stale)
broker_section = driver.find_element(
    By.XPATH, "//p[contains(text(), 'Broker Summary')]/ancestor::div[2]"
)

# Baru re-find date_button dari broker_section yang fresh
date_button = broker_section.find_element(
    By.XPATH,
    ".//button[@aria-label='Previous day']/following-sibling::button[1]"
)
driver.execute_script("arguments[0].click();", date_button)
time.sleep(1)
print("Berhasil klik tanggal.")

Ketemu cell tanggal 3, teks: 3


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":".//button[@aria-label='Previous day']/following-sibling::button[1]"}
  (Session info: MicrosoftEdge=150.0.4078.48); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	msedgedriver!GetHandleVerifier [0x7ff7c22d7135+e415]
	msedgedriver!GetHandleVerifier [0x7ff7c22d7194+e474]
	msedgedriver!GetHandleVerifier [0x7ff7c2956756+68da36]
	msedgedriver!(No symbol) [0x7ff7c1cd0762]
	msedgedriver!(No symbol) [0x7ff7c1cd09d5]
	msedgedriver!(No symbol) [0x7ff7c1cc740c]
	msedgedriver!(No symbol) [0x7ff7c1cc73b7]
	msedgedriver!(No symbol) [0x7ff7c1cc72bd]
	msedgedriver!(No symbol) [0x7ff7c1cc73b7]
	msedgedriver!(No symbol) [0x7ff7c1d0c2a8]
	msedgedriver!(No symbol) [0x7ff7c1cc6bfc]
	msedgedriver!(No symbol) [0x7ff7c1cc5e56]
	msedgedriver!(No symbol) [0x7ff7c1cc6a23]
	msedgedriver!(No symbol) [0x7

In [ ]:
# Khusus Scrapp

import pandas as pd
import base64
import os
from datetime import datetime

def _parse_value(val):
    """Parse nilai seperti '213.1B', '847.9M', '605K', atau angka biasa → float dalam satuan Miliar"""
    val = str(val).replace(",", "").strip()
    try:
        if "B" in val:
            return float(val.replace("B", ""))
        elif "M" in val:
            return float(val.replace("M", "")) / 1000
        elif "K" in val:
            return float(val.replace("K", "")) / 1_000_000
        else:
            return float(val)
    except:
        return 0.0

def scrape_hari_ini():
    print("Menunggu data tabel termuat...")
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, "tbody.ant-table-tbody tr.ant-table-row")
        )
    )
    time.sleep(2.5)  # Beri waktu render stabil

    rows = driver.find_elements(
        By.CSS_SELECTOR, "tbody.ant-table-tbody tr.ant-table-row"
    )

    buy_data  = {}
    sell_data = {}
    parsed_rows = []

    for row in rows:
        cols = row.find_elements(By.TAG_NAME, "td")
        if len(cols) < 8:
            continue

        broker_buy  = cols[0].text.strip()
        val_buy     = cols[1].text.strip()
        vol_buy     = cols[2].text.strip()
        avg_buy     = cols[3].text.strip()

        broker_sell = cols[4].text.strip()
        val_sell    = cols[5].text.strip()
        vol_sell    = cols[6].text.strip()
        avg_sell    = cols[7].text.strip()

        parsed_rows.append({
            "buy_broker": broker_buy,
            "buy_val": val_buy,
            "buy_vol": vol_buy,
            "buy_avg": avg_buy,
            "sell_broker": broker_sell,
            "sell_val": val_sell,
            "sell_vol": vol_sell,
            "sell_avg": avg_sell
        })

        if broker_buy and broker_buy != "-" and len(broker_buy) == 2:
            buy_data[broker_buy] = buy_data.get(broker_buy, 0) + _parse_value(val_buy)

        if broker_sell and broker_sell != "-" and len(broker_sell) == 2:
            sell_data[broker_sell] = sell_data.get(broker_sell, 0) + _parse_value(val_sell)

    all_brokers = set(buy_data) | set(sell_data)
    net_data = {}
    for broker in all_brokers:
        b = buy_data.get(broker, 0)
        s = sell_data.get(broker, 0)
        net_data[broker] = round(b - s, 4)

    return parsed_rows, net_data

def generate_validation_html(saham, parsed_rows, net_data, screenshot_path=None):
    screenshot_base64 = ""
    if screenshot_path and os.path.exists(screenshot_path):
        try:
            with open(screenshot_path, "rb") as image_file:
                screenshot_base64 = base64.b64encode(image_file.read()).decode('utf-8')
        except Exception as e:
            print(f"Gagal mengonversi screenshot ke base64: {e}")

    rows_html = ""
    for r in parsed_rows:
        rows_html += f"""
        <tr>
            <td class="buy-col font-mono font-bold">{r['buy_broker']}</td>
            <td class="buy-col text-right">{r['buy_val']}</td>
            <td class="buy-col text-right text-muted">{r['buy_vol']}</td>
            <td class="buy-col text-right text-muted">{r['buy_avg']}</td>
            <td class="divider-col"></td>
            <td class="sell-col font-mono font-bold">{r['sell_broker']}</td>
            <td class="sell-col text-right">{r['sell_val']}</td>
            <td class="sell-col text-right text-muted">{r['sell_vol']}</td>
            <td class="sell-col text-right text-muted">{r['sell_avg']}</td>
        </tr>
        """

    sorted_net = sorted(net_data.items(), key=lambda item: item[1], reverse=True)
    net_rows_html = ""
    for broker, net_val in sorted_net:
        net_class = "text-green" if net_val > 0 else ("text-red" if net_val < 0 else "")
        net_sign = "+" if net_val > 0 else ""
        net_rows_html += f"""
        <tr>
            <td class="font-mono font-bold">{broker}</td>
            <td class="text-right {net_class} font-semibold">{net_sign}{net_val:.4f} B</td>
        </tr>
        """

    screenshot_html = f"<img class='screenshot-img' src='data:image/png;base64,{screenshot_base64}' alt='Stockbit Screenshot'>" if screenshot_base64 else "<p class='no-screenshot'>Screenshot tidak tersedia</p>"

    scraped_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    html_content = f"""<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Notudo - Broker Summary Scraper Validation</title>
    <link href="https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@300;400;500;600;700;800&display=swap" rel="stylesheet">
    <style>
        :root {{
            --bg-main: #0b0f19;
            --bg-card: #151b2c;
            --border-color: #242f49;
            --text-main: #f3f4f6;
            --text-muted: #9ca3af;
            --primary: #6366f1;
            --green: #10b981;
            --red: #ef4444;
        }}
        * {{ box-sizing: border-box; margin: 0; padding: 0; }}
        body {{
            font-family: 'Plus Jakarta Sans', sans-serif;
            background-color: var(--bg-main);
            color: var(--text-main);
            padding: 2rem;
            line-height: 1.5;
        }}
        header {{
            margin-bottom: 2rem;
            display: flex;
            justify-content: space-between;
            align-items: center;
            border-bottom: 1px solid var(--border-color);
            padding-bottom: 1rem;
        }}
        h1 {{
            font-size: 1.75rem;
            font-weight: 700;
            background: linear-gradient(135deg, #6366f1, #a855f7);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
        }}
        .meta {{
            display: flex;
            gap: 1.5rem;
            font-size: 0.875rem;
            background-color: var(--bg-card);
            border: 1px solid var(--border-color);
            padding: 0.75rem 1.25rem;
            border-radius: 8px;
        }}
        .meta-item {{ display: flex; flex-direction: column; }}
        .meta-label {{
            color: var(--text-muted);
            font-size: 0.75rem;
            text-transform: uppercase;
            letter-spacing: 0.05em;
        }}
        .meta-value {{ font-weight: 600; }}
        .container {{
            display: grid;
            grid-template-columns: 1.2fr 1fr;
            gap: 2rem;
        }}
        @media (max-width: 1200px) {{
            .container {{ grid-template-columns: 1fr; }}
        }}
        .card {{
            background-color: var(--bg-card);
            border: 1px solid var(--border-color);
            border-radius: 12px;
            padding: 1.5rem;
            box-shadow: 0 4px 20px rgba(0, 0, 0, 0.25);
            display: flex;
            flex-direction: column;
            gap: 1.25rem;
        }}
        .card-title {{
            font-size: 1.2rem;
            font-weight: 600;
            border-bottom: 1px solid var(--border-color);
            padding-bottom: 0.75rem;
        }}
        table {{ width: 100%; border-collapse: collapse; font-size: 0.85rem; }}
        th {{
            text-align: left;
            padding: 0.5rem 0.75rem;
            color: var(--text-muted);
            font-weight: 600;
            border-bottom: 2px solid var(--border-color);
        }}
        td {{ padding: 0.5rem 0.75rem; border-bottom: 1px solid var(--border-color); }}
        tr:hover {{ background-color: rgba(255, 255, 255, 0.02); }}
        .buy-col {{ border-left: 3px solid rgba(16, 185, 129, 0.3); }}
        .sell-col {{ border-left: 3px solid rgba(239, 68, 68, 0.3); }}
        .divider-col {{ width: 15px; background-color: transparent; border: none; }}
        .text-right {{ text-align: right; }}
        .font-mono {{ font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace; }}
        .font-bold {{ font-weight: 700; }}
        .font-semibold {{ font-weight: 600; }}
        .text-muted {{ color: var(--text-muted); font-size: 0.8rem; }}
        .text-green {{ color: var(--green); }}
        .text-red {{ color: var(--red); }}
        .net-table-container {{
            max-height: 400px;
            overflow-y: auto;
            border: 1px solid var(--border-color);
            border-radius: 8px;
        }}
        .screenshot-container {{
            display: flex;
            justify-content: center;
            align-items: center;
            border: 1px solid var(--border-color);
            border-radius: 8px;
            overflow: hidden;
            background-color: #0b0f19;
            min-height: 300px;
            padding: 0.5rem;
        }}
        .screenshot-img {{ max-width: 100%; height: auto; display: block; border-radius: 4px; }}
        .no-screenshot {{ color: var(--text-muted); font-style: italic; }}
    </style>
</head>
<body>
    <header>
        <div>
            <h1>Notudo Scraper Validation</h1>
            <p style="color: var(--text-muted); font-size: 0.9rem; margin-top: 0.25rem;">Manual validation report for Stockbit Broker Summary scraper</p>
        </div>
        <div class="meta">
            <div class="meta-item">
                <span class="meta-label">Stock Ticker</span>
                <span class="meta-value text-green">{saham}</span>
            </div>
            <div class="meta-item">
                <span class="meta-label">Scraped At</span>
                <span class="meta-value">{scraped_at}</span>
            </div>
        </div>
    </header>

    <div class="container">
        <div style="display: flex; flex-direction: column; gap: 2rem;">
            <div class="card">
                <div class="card-title">Parsed Raw Rows (Buy vs Sell)</div>
                <div style="overflow-x: auto;">
                    <table>
                        <thead>
                            <tr>
                                <th colspan="4" class="text-green" style="border-bottom: 2px solid var(--green)">BUY SIDE</th>
                                <th></th>
                                <th colspan="4" class="text-red" style="border-bottom: 2px solid var(--red)">SELL SIDE</th>
                            </tr>
                            <tr>
                                <th>Broker</th>
                                <th class="text-right">Val</th>
                                <th class="text-right">Vol</th>
                                <th class="text-right">Avg</th>
                                <th></th>
                                <th>Broker</th>
                                <th class="text-right">Val</th>
                                <th class="text-right">Vol</th>
                                <th class="text-right">Avg</th>
                            </tr>
                        </thead>
                        <tbody>
                            {rows_html}
                        </tbody>
                    </table>
                </div>
            </div>

            <div class="card">
                <div class="card-title">Calculated Net Value per Broker</div>
                <div class="net-table-container">
                    <table>
                        <thead>
                            <tr>
                                <th>Broker</th>
                                <th class="text-right">Net Value</th>
                            </tr>
                        </thead>
                        <tbody>
                            {net_rows_html}
                        </tbody>
                    </table>
                </div>
            </div>
        </div>

        <div class="card" style="height: fit-content;">
            <div class="card-title">Stockbit Browser Screenshot (Original View)</div>
            <div class="screenshot-container">
                {screenshot_html}
            </div>
        </div>
    </div>
</body>
</html>
"""
    output_filename = f"scraped_validation_{saham.split('.')[0]}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.html"
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(html_content)
    print(f"\n[SUKSES] File HTML hasil scrape dibuat: {output_filename}")
    print("Silakan buka file tersebut untuk melakukan cek manual.")

# --- Jalankan Proses Scrape ---
try:
    parsed_rows, net_data = scrape_hari_ini()

    # Ambil screenshot dari section Broker Summary (pakai broker_section dari cell sebelumnya)
    screenshot_path = "temp_broker_summary.png"
    try:
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", broker_section)
        time.sleep(1.0)
        broker_section.screenshot(screenshot_path)
        print("Berhasil mengambil screenshot section Broker Summary.")
    except Exception as e:
        print(f"Gagal screenshot elemen: {e}. Mengambil screenshot halaman penuh...")
        try:
            driver.save_screenshot(screenshot_path)
        except:
            screenshot_path = None

    # Buat HTML
    generate_validation_html(SAHAM, parsed_rows, net_data, screenshot_path)

    # Bersihkan file screenshot temporary
    if screenshot_path and os.path.exists(screenshot_path):
        try:
            os.remove(screenshot_path)
        except:
            pass

except Exception as e:
    print(f"Gagal melakukan scraping: {e}")

Menunggu data tabel termuat...
Berhasil mengambil screenshot section Broker Summary.

[SUKSES] File HTML hasil scrape dibuat: scraped_validation_BBCA_20260704_143058.html
Silakan buka file tersebut untuk melakukan cek manual.
